In [12]:
from dotenv import load_dotenv
import os

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

from langchain.agents import create_agent
from langchain_core.tools import tool

from langgraph.checkpoint.memory import InMemorySaver

In [13]:
load_dotenv()

print("OPENAI_API_KEY chargé :", os.getenv("OPENAI_API_KEY") is not None)

OPENAI_API_KEY chargé : True


In [14]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

In [15]:
loader = PyPDFLoader("doc.pdf")
docs = loader.load()

print("Nombre de pages chargées :", len(docs))
print(docs[0].page_content[:500])

Nombre de pages chargées : 2
Smart City : Surveillance intelligente de la qualit´ e de l’air
Aya Fadel Najoua Mouaddab Hiba Zbari
Groupe 3
1 Mat´ eriel utilis´ e
Mat´ eriel Quantit´ e R´ ef´ erence / Mod` elePhoto
Raspberry Pi 4 1 Raspberry Pi 4 Model B
Carte micro-SD (32 GB) 1 SanDisk Ultra 32GB
Alimentation Raspberry
Pi
1 Official Raspberry Pi
Power Supply 5V/3A
Capteur MQ-135 1 MQ-135 Air Quality Gas
Sensor Module
Capteur PM2.5
(PMS5003)
1 Plantower PMS5003 Dust
Sensor
1


In [16]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(docs)

print("Nombre de chunks :", len(chunks))
print(chunks[0].page_content[:500])

Nombre de chunks : 2
Smart City : Surveillance intelligente de la qualit´ e de l’air
Aya Fadel Najoua Mouaddab Hiba Zbari
Groupe 3
1 Mat´ eriel utilis´ e
Mat´ eriel Quantit´ e R´ ef´ erence / Mod` elePhoto
Raspberry Pi 4 1 Raspberry Pi 4 Model B
Carte micro-SD (32 GB) 1 SanDisk Ultra 32GB
Alimentation Raspberry
Pi
1 Official Raspberry Pi
Power Supply 5V/3A
Capteur MQ-135 1 MQ-135 Air Quality Gas
Sensor Module
Capteur PM2.5
(PMS5003)
1 Plantower PMS5003 Dust
Sensor
1


In [17]:
embeddings = OpenAIEmbeddings()

In [18]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="chroma_db"
)

In [19]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [20]:
query = "De quoi parle ce document ?"
results = retriever.invoke(query)

for i, doc in enumerate(results, 1):
    print(f"\n--- Résultat {i} ---")
    print(doc.page_content[:500])


--- Résultat 1 ---
Smart City : Surveillance intelligente de la qualit´ e de l’air
Aya Fadel Najoua Mouaddab Hiba Zbari
Groupe 3
1 Mat´ eriel utilis´ e
Mat´ eriel Quantit´ e R´ ef´ erence / Mod` elePhoto
Raspberry Pi 4 1 Raspberry Pi 4 Model B
Carte micro-SD (32 GB) 1 SanDisk Ultra 32GB
Alimentation Raspberry
Pi
1 Official Raspberry Pi
Power Supply 5V/3A
Capteur MQ-135 1 MQ-135 Air Quality Gas
Sensor Module
Capteur PM2.5
(PMS5003)
1 Plantower PMS5003 Dust
Sensor
1

--- Résultat 2 ---
Mat´ eriel Quantit´ e R´ ef´ erence / Mod` elePhoto
Capteur DHT22 1 DHT22 / AM2302 Tem-
perature Humidity Sensor
Convertisseur ADC
(MCP3008)
1 MCP3008 10-bit Analog
to Digital Converter
Breadboard 1 MB-102 Solderless Bread-
board
Fils Dupont 20–30 Dupont Jumper Wires
(Male-Male)
R´ esistance 10kΩ 1 Carbon Film Resistor
10kΩ
Boˆ ıtier 1 Official Raspberry Pi 4
Case
2


In [21]:
@tool
def retrieve_documents(question: str) -> str:
    """Recherche les passages les plus pertinents dans les documents."""
    
    docs = retriever.invoke(question)

    if not docs:
        return "Aucune information pertinente trouvée dans les documents."

    context = "\n\n".join([doc.page_content for doc in docs])
    return context

In [22]:
memory = InMemorySaver()

In [23]:
agent_rag = create_agent(
    model=llm,
    tools=[retrieve_documents],
    checkpointer=memory,
    system_prompt=(
        "Tu es un chatbot RAG spécialisé dans les documents fournis. "
        "Quand l'utilisateur pose une question sur le contenu du document, "
        "utilise le tool retrieve_documents pour retrouver le contexte pertinent. "
        "Réponds uniquement à partir des informations extraites. "
        "Si l'information n'existe pas dans le document, dis clairement que tu ne l'as pas trouvée."
    )
)

In [24]:
resp = agent_rag.invoke(
    {
        "messages": [
            {"role": "user", "content": "Résume le document"}
        ]
    },
    config={"configurable": {"thread_id": "session1"}}
)

print(resp["messages"][-1].content)

Le document présente un projet de surveillance intelligente de la qualité de l'air, impliquant l'utilisation de divers matériels. Voici un résumé des composants utilisés :

1. **Raspberry Pi 4** - 1 unité (Modèle B)
2. **Carte micro-SD (32 GB)** - 1 unité (SanDisk Ultra 32GB)
3. **Alimentation Raspberry Pi** - 1 unité (5V/3A)
4. **Capteur MQ-135** - 1 unité (Module de capteur de gaz de qualité de l'air)
5. **Capteur PM2.5 (PMS5003)** - 1 unité (Capteur de poussière)
6. **Capteur DHT22** - 1 unité (Capteur de température et d'humidité)
7. **Convertisseur ADC (MCP3008)** - 1 unité (Convertisseur analogique-numérique 10 bits)
8. **Breadboard** - 1 unité (MB-102)
9. **Fils Dupont** - 20 à 30 unités (Fils male-male)
10. **Résistance 10kΩ** - 1 unité (Résistance en film carbone)
11. **Boîtier** - 1 unité (Boîtier officiel pour Raspberry Pi 4)

Ce projet semble viser à surveiller et analyser la qualité de l'air à l'aide de ces capteurs et équipements.


In [25]:
resp1 = agent_rag.invoke(
    {
        "messages": [
            {"role": "user", "content": "Je travaille sur ce document pour mon projet."}
        ]
    },
    config={"configurable": {"thread_id": "session1"}}
)

resp2 = agent_rag.invoke(
    {
        "messages": [
            {"role": "user", "content": "Quel est le sujet de notre conversation ?"}
        ]
    },
    config={"configurable": {"thread_id": "session1"}}
)

print(resp2["messages"][-1].content)

Le sujet de notre conversation porte sur un projet intitulé "Smart City : Surveillance intelligente de la qualité de l'air". Ce projet implique l'utilisation de divers matériels, notamment un Raspberry Pi 4 et plusieurs capteurs pour surveiller la qualité de l'air. Si vous avez des questions ou des points spécifiques à aborder concernant ce projet, n'hésitez pas à les poser !
